In [32]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import ipywidgets as widgets
from IPython.display import display, clear_output

# -----------------------------
# Linear algebra helpers
# -----------------------------
ket00 = np.array([1, 0, 0, 0], dtype=complex)

basis_labels = ["|00>", "|01>", "|10>", "|11>"]

def build_ry(theta: float) -> np.ndarray:
    c = np.cos(theta / 2.0)
    s = np.sin(theta / 2.0)
    return np.array([[c, -s],
                     [s,  c]], dtype=complex)

def kron(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.kron(a, b)

def apply_single_qubit_gate_two_qubits(gate: np.ndarray, qubit: int) -> np.ndarray:
    if qubit == 0:
        return kron(gate, np.eye(2, dtype=complex))
    elif qubit == 1:
        return kron(np.eye(2, dtype=complex), gate)
    raise ValueError("qubit must be 0 or 1")

def build_cnot_01() -> np.ndarray:
    # control qubit 0, target qubit 1
    return np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0]
    ], dtype=complex)

CNOT_01 = build_cnot_01()

# -----------------------------
# Target state family
# -----------------------------
def concurrence_to_theta(concurrence: float) -> float:
    c = np.clip(concurrence, 0.0, 1.0)
    return 0.5 * np.arcsin(c)

def build_target_state_from_concurrence(concurrence: float) -> np.ndarray:
    theta = concurrence_to_theta(concurrence)
    state = np.array([
        np.cos(theta),
        0.0,
        0.0,
        np.sin(theta)
    ], dtype=complex)
    return normalize_state(state)

# -----------------------------
# Ansatz
# -----------------------------
def build_ansatz_state(alpha: float) -> np.ndarray:
    ry0 = apply_single_qubit_gate_two_qubits(build_ry(alpha), qubit=0)
    state = ry0 @ ket00
    state = CNOT_01 @ state
    return normalize_state(state)

# -----------------------------
# Metrics
# -----------------------------
def normalize_state(state: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(state)
    if norm == 0:
        raise ValueError("Zero-norm state encountered.")
    return state / norm

def state_fidelity(state_a: np.ndarray, state_b: np.ndarray) -> float:
    overlap = np.vdot(state_a, state_b)
    return float(np.abs(overlap) ** 2)

def pure_state_concurrence_two_qubit(state: np.ndarray) -> float:
    # For |psi> = [a,b,c,d], concurrence = 2|ad - bc|
    a, b, c, d = state
    return float(2.0 * np.abs(a * d - b * c))

def cost_function_alpha(alpha_array: np.ndarray, target_state: np.ndarray) -> float:
    alpha = float(alpha_array[0])
    trial_state = build_ansatz_state(alpha)
    fidelity = state_fidelity(trial_state, target_state)
    return 1.0 - fidelity

# -----------------------------
# Pretty summary helpers
# -----------------------------
def format_state_vector(state: np.ndarray, decimals: int = 6) -> str:
    pieces = []
    for amp, label in zip(state, basis_labels):
        re_part = np.round(amp.real, decimals)
        im_part = np.round(amp.imag, decimals)
        if abs(im_part) < 10**(-decimals):
            pieces.append(f"{re_part}{label}")
        else:
            pieces.append(f"({re_part}+{im_part}j){label}")
    return " + ".join(pieces)

def extract_real_amplitudes_for_plot(state: np.ndarray) -> np.ndarray:
    # For this demo states are real-valued, but keep explicit conversion
    return np.real(state)

In [33]:
def run_hybrid_optimizer(
    target_concurrence: float,
    random_init: bool = True,
    alpha_init: float = 0.1,
    maxiter: int = 100,
    seed: int = 0
) -> dict:
    rng = np.random.default_rng(seed)

    target_state = build_target_state_from_concurrence(target_concurrence)

    if random_init:
        alpha0 = rng.uniform(-2.0 * np.pi, 2.0 * np.pi)
    else:
        alpha0 = float(alpha_init)

    trace_iter = []
    trace_cost = []
    trace_alpha = []
    trace_fidelity = []
    trace_concurrence = []

    def callback(xk):
        alpha = float(xk[0])
        trial_state = build_ansatz_state(alpha)
        fidelity = state_fidelity(trial_state, target_state)
        cost = 1.0 - fidelity
        conc = pure_state_concurrence_two_qubit(trial_state)

        trace_iter.append(len(trace_iter))
        trace_cost.append(cost)
        trace_alpha.append(alpha)
        trace_fidelity.append(fidelity)
        trace_concurrence.append(conc)

    result = minimize(
        fun=cost_function_alpha,
        x0=np.array([alpha0], dtype=float),
        args=(target_state,),
        method="Nelder-Mead",
        callback=callback,
        options={"maxiter": int(maxiter), "xatol": 1e-10, "fatol": 1e-12, "disp": False}
    )

    alpha_opt = float(result.x[0])
    final_state = build_ansatz_state(alpha_opt)
    final_fidelity = state_fidelity(final_state, target_state)
    final_concurrence = pure_state_concurrence_two_qubit(final_state)

    # Make sure at least final point is recorded
    if len(trace_iter) == 0 or trace_alpha[-1] != alpha_opt:
        trace_iter.append(len(trace_iter))
        trace_cost.append(1.0 - final_fidelity)
        trace_alpha.append(alpha_opt)
        trace_fidelity.append(final_fidelity)
        trace_concurrence.append(final_concurrence)

    return {
        "target_concurrence": float(target_concurrence),
        "target_state": target_state,
        "target_theta": concurrence_to_theta(target_concurrence),
        "alpha_init": float(alpha0),
        "alpha_opt": alpha_opt,
        "final_state": final_state,
        "final_fidelity": final_fidelity,
        "final_concurrence": final_concurrence,
        "success": bool(result.success),
        "message": result.message,
        "nfev": int(result.nfev),
        "nit": int(result.nit) if hasattr(result, "nit") else None,
        "trace_iter": np.array(trace_iter),
        "trace_cost": np.array(trace_cost),
        "trace_alpha": np.array(trace_alpha),
        "trace_fidelity": np.array(trace_fidelity),
        "trace_concurrence": np.array(trace_concurrence),
    }

def plot_results_square(result: dict) -> None:
    # Prepare amplitude comparison
    target_amp = extract_real_amplitudes_for_plot(result["target_state"])
    final_amp = extract_real_amplitudes_for_plot(result["final_state"])

    x = np.arange(len(basis_labels))
    width = 0.38

    # Three compact square panels in one row
    fig, axes = plt.subplots(1, 3, figsize=(12.0, 4.0), constrained_layout=True)
    ax1, ax2, ax3 = axes

    # Plot 1: cost trace
    ax1.plot(result["trace_iter"], result["trace_cost"], marker="o", markersize=4)
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Cost = 1 - Fidelity")
    ax1.set_title("Optimization Trace")
    ax1.grid(True, alpha=0.3)
    ax1.set_box_aspect(1)

    # Plot 2: parameter stabilization
    ax2.plot(result["trace_iter"], result["trace_alpha"], marker="o", markersize=4)
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel(r"Parameter $\alpha$")
    ax2.set_title("Parameter Stabilization")
    ax2.grid(True, alpha=0.3)
    ax2.set_box_aspect(1)

    # Plot 3: amplitude comparison
    ax3.bar(x - width / 2, target_amp, width=width, label="Target")
    ax3.bar(x + width / 2, final_amp, width=width, label="Optimized")
    ax3.set_xticks(x)
    ax3.set_xticklabels(basis_labels)
    ax3.set_ylabel("Amplitude")
    ax3.set_title("Target vs Optimized")
    ax3.legend(fontsize=8)
    ax3.grid(True, axis="y", alpha=0.3)
    ax3.set_box_aspect(1)

    plt.show()

    print("Final numerical verification")
    print("-" * 100)

    left_lines = [
        f"Target concurrence  : {result['target_concurrence']:.6f}",
        f"Final concurrence   : {result['final_concurrence']:.6f}",
        f"Final fidelity      : {result['final_fidelity']:.12f}",
        f"Initial alpha       : {result['alpha_init']:.6f}",
        f"Optimized alpha     : {result['alpha_opt']:.12f}",
        f"Optimizer success   : {result['success']}",
        #f"Optimizer message   : {result['message']}",
    ]

    right_lines = [
        f"Function evals      : {result['nfev']}",
        f"Iterations          : {result['nit']}",
        "Target state:",
        *format_state_vector(result["target_state"]).splitlines(),
        "Optimized state:",
        *format_state_vector(result["final_state"]).splitlines(),
    ]

    col_width = 52
    n_rows = max(len(left_lines), len(right_lines))

    for i in range(n_rows):
        left = left_lines[i] if i < len(left_lines) else ""
        right = right_lines[i] if i < len(right_lines) else ""
        print(f"{left:<{col_width}}{right}")
    

In [34]:
def build_optimizer_ui():
    concurrence_slider = widgets.FloatSlider(
        value=1.0,
        min=0.0,
        max=1.0,
        step=0.01,
        description="Target C:",
        continuous_update=False,
        readout_format=".2f",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="500px")
    )

    random_init_checkbox = widgets.Checkbox(
        value=True,
        description="Random initial guess"
    )

    alpha_init_slider = widgets.FloatSlider(
        value=0.10,
        min=-2.0 * np.pi,
        max=2.0 * np.pi,
        step=0.05,
        description="Initial α:",
        continuous_update=False,
        readout_format=".2f",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="500px")
    )

    seed_int = widgets.IntText(
        value=0,
        description="Seed:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="220px")
    )

    maxiter_int = widgets.IntText(
        value=100,
        description="Max iter:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="220px")
    )

    run_button = widgets.Button(
        description="Run Hybrid Optimizer",
        button_style="primary"
    )

    output = widgets.Output()

    def on_run_clicked(_):
        with output:
            clear_output(wait=True)

            result = run_hybrid_optimizer(
                target_concurrence=concurrence_slider.value,
                random_init=random_init_checkbox.value,
                alpha_init=alpha_init_slider.value,
                maxiter=maxiter_int.value,
                seed=seed_int.value
            )

            plot_results_square(result)

    run_button.on_click(on_run_clicked)

    controls = widgets.VBox([
        widgets.HTML("<h3>Hybrid Optimizer for Target-State Preparation</h3>"),
        concurrence_slider,
        random_init_checkbox,
        alpha_init_slider,
        widgets.HBox([seed_int, maxiter_int]),
        run_button
    ])

    ui = widgets.VBox([controls, output])
    return ui

In [35]:
ui = build_optimizer_ui()
display(ui)